In [ ]:
import sys, os 
import geopandas as gpd 
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
data_path = os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon.geojson')
data = gpd.read_file(data_path)

In [2]:
data

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,geometry
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ..."
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ..."
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ..."
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ..."
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1..."
...,...,...,...,...,...,...,...,...,...
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3...."
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3..."
1952,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,4458.659,"MULTIPOLYGON (((-75.12605 3.81572, -75.12607 3..."
1953,None,None,None,None,None,None,None,NaN,None


### 1: Build Image Download URL 

Since the datasets are coming from different sources and all of them are independent drone images, we have 30 distinct images sitting in different locations with different IDs. We only had the links to those datasets which are not the download links, hence we need to go to STAC if it is OAM and go to Google Drive if it is gdrive. We also need to distinguish the type of the datasets! Hence, the first step is to write logic that would help us build the image download URL and identify the type of sources for drone images. When we say "type of source" we mean either OAM or Google Drive.

In [3]:
from src.preprocess import build_image_download_uri 

In [ ]:
gdf = await build_image_download_uri(data,'OAM')

### 2: Build Unique ID for Each Polygon 

Just to distinguish them and make them consistent throughout the experiment, and also helpful for debugging, we will build a unique ID for the polygons.

In [5]:
gdf['poly_id'] = range(len(gdf))

In [ ]:
gdf = gdf[gdf['OAM'].notna() & (gdf['OAM'] != '')]  # drop null rows without images

In [9]:
len(gdf)

1953

### 3: Build Unique IDs for Images 

To avoid the possibility of duplications while downloading the images, we have observed that multiple polygon features have references to the same drone image, meaning a many-to-one relationship! Hence, to avoid downloading the same image twice, we needed to know the unique image UID and assign it to the respective columns, which will also help us later to stitch them back with the polygons!

In [10]:
from src.preprocess import assign_image_uids 

In [ ]:
gdf, _ = assign_image_uids(gdf, 'download_url')

In [ ]:
print(gdf['image_uid'].nunique(), gdf['download_url'].nunique())

30 30


We observe that there are 30 unique images and they align with the download URL count, so our image UID generation logic is correct.

In [28]:
gdrive_df = gdf[gdf['url_type'] == 'gdrive'][['image_uid', 'download_url']].drop_duplicates()
gdrive_gdf = dict(zip(gdrive_df['image_uid'], gdrive_df['download_url']))

### 4: Cleanup the Google Drive Download Links 

The Google Drive links shared are of folders, and the script can't go through the folder to determine the TIFF - it would be complicated. Hence, we do a manual mapping for those images with Google Drive links to correctly place the image download link instead of the folder. Since there were only 7, we can do it manually, and we already know the image UID so we can replace them in the original dataframe with the mapped URLs.

In [29]:
gdrive_gdf

{'img_0004_befb8cbc': 'https://drive.google.com/drive/folders/1QRkuTFi_F-_uvWXQMtEvoUKSD_T12-7F?usp=drive_link',
 'img_0005_b3f008c4': 'https://drive.google.com/drive/folders/1LXcT0lrf1_MqfedpQHI6tf_qBJxd03VQ?usp=drive_link',
 'img_0014_0aad0f9c': 'https://drive.google.com/drive/folders/1PoaKqUvMOJI90EOKiKBlT5OXgkUYh-kQ?usp=sharing',
 'img_0015_0f954bee': 'https://drive.google.com/drive/folders/19TyWiBf5cd0-OTJbFz0G5qLoQL3Xga-0?usp=drive_link',
 'img_0016_8cb9e2ca': 'https://drive.google.com/drive/folders/1tMNj7BR3uq20TUqxDsnZQihXJRPNakxJ?usp=drive_link',
 'img_0017_f8406eda': 'https://drive.google.com/drive/folders/14rikAgD3b-cd_lmgeXK7qmHKB62qY0sv?usp=drive_link',
 'img_0018_3c5c5a9c': 'https://drive.google.com/drive/folders/1e1e--U2iTxCx-of2m2HWiGEAnMUivtMm?usp=drive_link'}

In [ ]:
gdrive_gdf_fixed = gdrive_gdf  # let's fix those bastard Google Drive links
gdrive_gdf_fixed['img_0004_befb8cbc'] = 'https://drive.google.com/file/d/1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT/view?usp=sharing'
gdrive_gdf_fixed['img_0005_b3f008c4'] = 'https://drive.google.com/file/d/1KrC1AXL5IcmGCj_havxxlzG_gnGDzjAv/view?usp=sharing'
gdrive_gdf_fixed['img_0014_0aad0f9c'] = 'https://drive.google.com/file/d/1T1Pin5cXsCcu42Ogx1apB-_2avYBwsv8/view?usp=sharing'
gdrive_gdf_fixed['img_0015_0f954bee'] = 'https://drive.google.com/file/d/1Pz9em-zdp8yNXEKZPUrtiFNJEl3MUq_8/view?usp=sharing'
gdrive_gdf_fixed['img_0016_8cb9e2ca'] = 'https://drive.google.com/file/d/1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J/view?usp=sharing'
gdrive_gdf_fixed['img_0017_f8406eda'] = 'https://drive.google.com/file/d/1skz0zSyMGh6R5WC8MVe2kzfPr4DGyc6O/view?usp=sharing'
gdrive_gdf_fixed['img_0018_3c5c5a9c'] = 'https://drive.google.com/file/d/1CqNSAEJhfVvpryHvwEy43HJi0dyn_94C/view?usp=sharing'

In [31]:
gdrive_gdf_fixed

{'img_0004_befb8cbc': 'https://drive.google.com/file/d/1uDznoDuGo5OsSnSd3JFxOvYB0rVuXCIT/view?usp=sharing',
 'img_0005_b3f008c4': 'https://drive.google.com/file/d/1KrC1AXL5IcmGCj_havxxlzG_gnGDzjAv/view?usp=sharing',
 'img_0014_0aad0f9c': 'https://drive.google.com/file/d/1T1Pin5cXsCcu42Ogx1apB-_2avYBwsv8/view?usp=sharing',
 'img_0015_0f954bee': 'https://drive.google.com/file/d/1Pz9em-zdp8yNXEKZPUrtiFNJEl3MUq_8/view?usp=sharing',
 'img_0016_8cb9e2ca': 'https://drive.google.com/file/d/1zPcUXSBDHLsMpPslu4VcT29XiIb2sB0J/view?usp=sharing',
 'img_0017_f8406eda': 'https://drive.google.com/file/d/1skz0zSyMGh6R5WC8MVe2kzfPr4DGyc6O/view?usp=sharing',
 'img_0018_3c5c5a9c': 'https://drive.google.com/file/d/1CqNSAEJhfVvpryHvwEy43HJi0dyn_94C/view?usp=sharing'}

In [32]:
gdf['download_url'] = gdf['image_uid'].map(gdrive_gdf_fixed).fillna(gdf['download_url'])

/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/.venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
gdf.to_file(os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon_cleaned.geojson'), driver='GeoJSON')

### 5: Download the Images 

Now that we have everything sorted for the download to begin, we start the download and make sure they are all in the same folder with the right image UID name rather than their original name, as they might mess up our link to the polygons.

In [35]:
from src.preprocess import download_images

In [ ]:
await download_images(gdf, os.path.join(os.getcwd(), '..', 'data', 'images'))

### 6: Standardize the Downloaded Images 

Some of the downloaded images are in COG format but some are in 16-bit. All of them should have 0-255 grayscale values, so to make all of them standard and enable us to mosaic them for visualization too, we convert them to Cloud Optimized GeoTIFF with 8-bit unsigned integers (values from 0 to 255) data type (just to bring all the images into the same data types and same gray levels). COG will enable those huge drone images to be tiled, so later on when we build the features for our model, the stats computation would be faster as we would avoid loading the entire images into memory and we can only query the polygon part in the big image.

We also created a mosaic to visualize them properly in QGIS without loading them and via tiles. Initially, we thought about what we would do when there is an overlap - we did observe that a single polygon might have two overlapping drone tiles, but we decided to use the image UID so we would avoid the overlap as we already know which drone image to use because it completely covers the polygon and which mapper used it, so no need to go with the mosaic option. It is still useful for tiling and visualization though.

In [ ]:
bash_script_path = os.path.join(os.getcwd(), '..', 'tif2cog.sh')

In [38]:
! bash $bash_script_path ../data/images

1/30 Converting: img_0026_40a8e8df.tif
2/30 Converting: img_0029_37df798b.tif
3/30 Converting: img_0027_cbc71305.tif
4/30 Converting: img_0023_d68997fa.tif
5/30 Converting: img_0020_4a8f0af8.tif
6/30 Converting: img_0000_c4a77f3e.tif
7/30 Converting: img_0008_4f1122ba.tif
8/30 Converting: img_0028_47d0c657.tif
9/30 Converting: img_0024_95971f64.tif
10/30 Converting: img_0001_f4dde9b0.tif
11/30 Converting: img_0017_f8406eda.tif
12/30 Converting: img_0013_08ee5eac.tif
13/30 Converting: img_0018_3c5c5a9c.tif
14/30 Converting: img_0006_2828f6be.tif
15/30 Converting: img_0009_c57f932c.tif
16/30 Converting: img_0012_f4d05bf8.tif
17/30 Converting: img_0019_2d8e19cb.tif
18/30 Converting: img_0014_0aad0f9c.tif
19/30 Converting: img_0016_8cb9e2ca.tif
20/30 Converting: img_0022_4e02d22a.tif
21/30 Converting: img_0025_4090d330.tif
22/30 Converting: img_0007_eec6a1e9.tif
23/30 Converting: img_0003_b06e8b27.tif
24/30 Converting: img_0005_b3f008c4.tif
25/30 Converting: img_0004_befb8cbc.tif
26/30 Con